**SOC612: Data Analytics for the Social Sciences**

**Date: September 15, 2026**

**Author: R. Duerr**

**Lecture 2: Working with data**

For the use of external data, we'll check and set a working directory, and import the data from the World Value Survey (Waves 6-7). We assign it the object name 'data'.

For .csv files, we can use pd.read_csv() from pandas. For common alternative formats you can use, for example: pd.read_csv(sep='\t') or pd.read_table() for .tsv files, pd.read_excel() from pandas, pd.read_json() from pandas, pd.read_xml() or pd.read_html() from pandas.

For files from other statistical programmes (e.g., SPSS, SAS, or Stata), you can use pyreadstat.read_sav() for SPSS, pyreadstat.read_dta() for Stata, and pyreadstat.read_sas7bdat() for SAS (all in pyreadstat).

In [5]:
import pandas as pd
import os

# Check and set the working directory
print(os.getcwd())  # Print current working directory

# Load the CSV file
data = pd.read_csv("wvs6_7.csv")

/content


/tmp/ipykernel_1624/2572795547.py:8: DtypeWarning: Columns (346,348,350,580,581,582,583) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("wvs6_7.csv")


Wave 6-7 from WVS has 613 variables. For our analysis, we are only interested in a few selected variables. Let's subset the full data into a simpler object. For our analysis on people's view on economic competition (Q106-Q111) and science and technology (Q158-Q163) in East Asian societies, and how they correlate with socio-demographics (Q260, Q262, Q273, Q275). We'll retain the original object. In Python, the pandas library (along with NumPy, Dask, or Polars) provides many essential data editing and manipulation functions.

After subsetting the selected variables, we do some checks on variable levels. While subsetting, we'll rename our selected variables.

In [6]:
# Select and rename columns
subset = data.rename(columns={
    'B_COUNTRY': 'country',
    'Q260': 'sex', 'Q262': 'age', 'Q273': 'marital_status','Q275': 'education',
    'Q106': 'eco1', 'Q107': 'eco2', 'Q108': 'eco3', 'Q109': 'eco4',
    'Q110': 'eco5', 'Q111': 'eco6',
    'Q158': 'sci1','Q159': 'sci2','Q160': 'sci3','Q161': 'sci4','Q162': 'sci5',
    'Q163': 'sci6'
})[['country', 'sex', 'age', 'marital_status', 'education',
   'eco1', 'eco2', 'eco3', 'eco4', 'eco5', 'eco6',
   'sci1', 'sci2', 'sci3', 'sci4', 'sci5', 'sci6']]

# Filter for East Asian societies
subset_EA = subset[subset['country'].isin([156, 446, 344, 392, 410, 158])].copy()

We have limited our dataset to 17 columns, and 9,955 rows, down from 613 columns and 97,220 rows. Let's continue with making it more handy to work with.

- Assign the correct variable levels: Some variables (country, sex, marital status, education) should be ordinal rather than metric/quasi-metric.

- Check the value ranges of the variables. Recode them to NA, where necessary.

In [7]:
# Recode country codes
country_mapping = {
    156: "China", 446: "Hong Kong SAR", 344: "Macao SAR", 392: "Japan",
    410: "South Korea", 158: "Taiwan ROC"
}
subset_EA['country'] = subset_EA['country'].map(country_mapping)
subset_EA['country'] = pd.Categorical(subset_EA['country'])

import numpy as np

# Sex
sex_mapping = {1: "male", 2: "female"}
subset_EA['sex'] = subset_EA['sex'].map(sex_mapping)
subset_EA['sex'] = pd.Categorical(subset_EA['sex'])

# Marital status
marital_mapping = {
    1: "married", 2: "married_cohabiting", 3: "divorced",
    4: "separated", 5: "widowed", 6: "single"
}
subset_EA['marital_status'] = subset_EA['marital_status'].map(marital_mapping)
subset_EA['marital_status'] = pd.Categorical(subset_EA['marital_status'])

# Simplify marital status
subset_EA['marital_status'] = subset_EA['marital_status'].apply(
    lambda x: "partnered" if x in ["married", "married_cohabiting"]
              else "single" if x in ["divorced", "separated", "widowed", "single"]
              else None
)
subset_EA['marital_status'] = pd.Categorical(subset_EA['marital_status'])

# Education
education_mapping = {
    0: "primary_or_below", 1: "primary_or_below", 2: "secondary",
    3: "secondary", 4: "post_secondary", 5: "post_secondary",
    6: "tertiary", 7: "tertiary", 8: "tertiary"
}
subset_EA['education'] = subset_EA['education'].map(education_mapping)
subset_EA['education'] = pd.Categorical(subset_EA['education'])

# Age
print(subset_EA['age'].min(), subset_EA['age'].max())
subset_EA.loc[subset_EA['age'] < 18, 'age'] = np.nan

# Economic values (columns 6-10: eco1-eco5)
eco_cols = ['eco1', 'eco2', 'eco3', 'eco4', 'eco5']
subset_EA[eco_cols] = subset_EA[eco_cols].apply(
    lambda col: col.where((col >= 1) & (col <= 10), np.nan)
)

# Science & technology values (columns 12-17: sci1-sci6)
sci_cols = ['sci1', 'sci2', 'sci3', 'sci4', 'sci5', 'sci6']
subset_EA[sci_cols] = subset_EA[sci_cols].apply(
    lambda col: col.where((col >= 1) & (col <= 10), np.nan)
)

# Recode eco6
eco6_mapping = {1: "environment", 2: "growth"}
subset_EA['eco6'] = subset_EA['eco6'].map(eco6_mapping)
subset_EA['eco6'] = pd.Categorical(subset_EA['eco6'])

-5 95


Assume we want to align the first four of the economic values variables and all six of the science and technology variables along a spectrum. For economic values, 'more collectivist' (1) vs 'more individualist' (10), for science and technology, 'more cautious' (1) vs 'more innovative' (10).

For now, we also just want to work with complete cases on these variables, so we'll remove cases with NA's on them first.

- Remove NA's on eco1-eco4, and sci1-sci6
- eco2 and eco4 have to be reversed.
- sci3, sci4, and sci5 have to be reversed.

In a second step, we want to standardize them. We will do two versions of it:

- We standardize them using the whole sample.
- We standardize them country-wise.

In [8]:
# Remove rows with NAs in eco1-eco4 and sci1-sci6
cols_to_check = ['eco1', 'eco2', 'eco3', 'eco4', 'sci1', 'sci2', 'sci3', 'sci4', 'sci5', 'sci6']
subset_EA = subset_EA.dropna(subset=cols_to_check)

# Reverse economic values (eco2, eco4)
subset_EA[['eco2', 'eco4']] = 11 - subset_EA[['eco2', 'eco4']]

# Reverse science and technology values (sci3, sci4, sci5)
subset_EA[['sci3', 'sci4', 'sci5']] = 11 - subset_EA[['sci3', 'sci4', 'sci5']]

# Sum of eco1, eco2, eco3, eco4
subset_EA['eco_sum'] = subset_EA[['eco1', 'eco2', 'eco3', 'eco4']].sum(axis=1)

# Standardize (z-score) the sum
subset_EA['eco_sum'] = (subset_EA['eco_sum'] - subset_EA['eco_sum'].mean()) / subset_EA['eco_sum'].std()

# Sum of sci1, sci2, sci3, sci4, sci5, sci6
subset_EA['sci_sum'] = subset_EA[['sci1', 'sci2', 'sci3', 'sci4', 'sci5', 'sci6']].sum(axis=1)

# Standardize (z-score) the sum
subset_EA['sci_sum'] = (subset_EA['sci_sum'] - subset_EA['sci_sum'].mean()) / subset_EA['sci_sum'].std()

# Sum of eco1, eco2, eco3, eco4 by row
subset_EA['eco_sum_cw'] = subset_EA[['eco1', 'eco2', 'eco3', 'eco4']].sum(axis=1)

# Standardize (z-score) the sum within each country
subset_EA['eco_sum_cw'] = subset_EA.groupby('country')['eco_sum_cw'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Sum of sci1, sci2, sci3, sci4, sci5, sci6 by row
subset_EA['sci_sum_cw'] = subset_EA[['sci1', 'sci2', 'sci3', 'sci4', 'sci5', 'sci6']].sum(axis=1)

# Standardize (z-score) the sum within each country
subset_EA['sci_sum_cw'] = subset_EA.groupby('country')['sci_sum_cw'].transform(
    lambda x: (x - x.mean()) / x.std()
)

/tmp/ipykernel_1624/3928432237.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  subset_EA['eco_sum_cw'] = subset_EA.groupby('country')['eco_sum_cw'].transform(
/tmp/ipykernel_1624/3928432237.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  subset_EA['sci_sum_cw'] = subset_EA.groupby('country')['sci_sum_cw'].transform(


In some cases, we might want to create composite indices which combine multiple variables. This can, however, complicate interpretations sometimes, and should usually only be done when it's theoretically well justified, or if it follows an exploratory analysis. Let's assume we did that for our example, and we want to create an Socio-economic index variable based on age, income, and marital status.

- Using age and education level, we create an additive index.

- We weigh it by marital status.

In [9]:
# Map education levels to numeric values
edu_levels = ["primary_or_below", "secondary", "post_secondary", "tertiary"]
subset_EA['edu_num'] = pd.Categorical(subset_EA['education'], categories=edu_levels, ordered=True).codes

# Assign marital weight
subset_EA['marital_weight'] = np.where(subset_EA['marital_status'] == "partnered", 1.0, 0.8)

# Calculate index
subset_EA['index'] = (subset_EA['age'] + subset_EA['edu_num']) * subset_EA['marital_weight']

In some cases, we might want to run multi-level models that involve structural variables besides individual level. Let's assume, for all of the countries in our dataset, we want to add the life expectancy (in 2017), based on gender. The World Bank has a data repository on many structural indicators.

- We load the second dataset (lifeexp.xlsx) into our environment. Since the working directoy is already set, we can load the data directly.

- We add the life-expectancies based on gender and country using a pd.merge(..., how='left').

In [10]:
# Read Excel file
lifeexp = pd.read_excel("lifeexp.xlsx")

# Select and rename columns
lifeexp_subset = lifeexp[["Country Name", "Disaggregation", "Value"]].rename(
    columns={"Value": "life_exp"}
)

# Merge with subset_EA
subset_EA = pd.merge(
    subset_EA,
    lifeexp_subset,
    left_on=["country", "sex"],
    right_on=["Country Name", "Disaggregation"],
    how="left"
)

There are cases where we might have to create our own specific function, because there is no built-in function in Python's standard library or third-party packages. Let's construct the following functions:

- Square function: a function that exponentiates a value by power 2 (^2); we'll use this to create an age square variable.

- Percentage function: a function that calculates the percentage of an observed value from the maximum value; we'll use this to create a variable that indicates how much a person has reached in terms of country- and gender-specific life expectancy.

- Conditional function: a function that indicates whether someone's country-specific standardized eco_sum_cw (economic values) and sci_sum_cw (science and technology) values exceeds the overall values, based on the following criteria: if the country-specific value is higher than the overall value, we assign a 1; if the country-specific value is more than twice as high as the overall value, we assign it a 2; for all other cases, we give them a 0.

In [11]:
import numpy as np

# 1. Square function
def sqf(x):
    return x ** 2

subset_EA['age_sq'] = sqf(subset_EA['age'])

# 2. Percentage function
def prct(part, total):
    return (part / total) * 100

subset_EA['age_pct'] = prct(subset_EA['age'], subset_EA['life_exp'])

# 3. Conditional values function
def scaled_values_comp(country_scaled, overall_scaled):
    return np.where(
        np.abs(country_scaled) < np.abs(overall_scaled), 0,
        np.where(
            np.abs(country_scaled) > 2 * np.abs(overall_scaled), 2, 1
        )
    )

subset_EA['eco_scaled_comp'] = scaled_values_comp(subset_EA['eco_sum_cw'], subset_EA['eco_sum'])
subset_EA['sci_scaled_comp'] = scaled_values_comp(subset_EA['sci_sum_cw'], subset_EA['sci_sum'])

If we want to iterate over a vector or list or another sequence of items, we can also use loops in R. Let's assume we want to mark the positive and negative extreme cases among the respondents. We identify those rows, where the respondents have either the full scores in both economic values (eco1-eco4) and science and technology values (sci1-sci6).

- We create a variable indicating whether a respondent scores 100 ("positive"), 10 ("negative") or any other number ("mixed").

In [12]:
import numpy as np

# Define the variables to sum
vars_to_sum = ["eco1", "eco2", "eco3", "eco4",
               "sci1", "sci2", "sci3", "sci4", "sci5", "sci6"]

subset_EA['outlier'] = None

# Loop through the rows
for i in range(len(subset_EA)):
    row_sum = subset_EA.iloc[i][vars_to_sum].sum()

    if row_sum == 100:
        subset_EA.at[i, 'outlier'] = "pos"
    elif row_sum == 10:
        subset_EA.at[i, 'outlier'] = "neg"
    else:
        subset_EA.at[i, 'outlier'] = "mixed"

subset_EA['outlier'] = pd.Categorical(subset_EA['outlier'])

Before getting into more complex exploratory analyses, let's do a quick descriptive check of the variables and the dataset to see whether all of our changes appear plausible. We have several options for simple descriptive functions, among them df.describe(), df.info(), or df.head() from pandas.

In [13]:
print(subset_EA.describe(include='all'))  # Summary statistics for all columns
print(subset_EA.info())                  # Structure of the dataset (similar to str())
print(subset_EA.head())  # Quick look at the first few rows (similar to glimpse)

       country     sex          age marital_status  education         eco1  \
count     9216    9215  8998.000000           9190       9161  9216.000000   
unique       6       2          NaN              2          4          NaN   
top      China  female          NaN      partnered  secondary          NaN   
freq      2919    4914          NaN           6309       4137          NaN   
mean       NaN     NaN    46.253056            NaN        NaN     6.097439   
std        NaN     NaN    16.000454            NaN        NaN     2.317606   
min        NaN     NaN    18.000000            NaN        NaN     1.000000   
25%        NaN     NaN    33.000000            NaN        NaN     5.000000   
50%        NaN     NaN    46.000000            NaN        NaN     6.000000   
75%        NaN     NaN    59.000000            NaN        NaN     8.000000   
max        NaN     NaN    95.000000            NaN        NaN    10.000000   

               eco2         eco3         eco4         eco5  ...